In [2]:
import warnings
warnings.simplefilter(action='ignore', category=Warning)
import os, sys
# to avoid any possible jupyter crashes due to rpy2 not finding the R install on conda
os.environ['R_HOME'] = sys.exec_prefix+"/lib/R/"
import scanpy as sc
import scFates as scf
sc.set_figure_params(fontsize=16)
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
adata = sc.read_h5ad("./adata/CM_Ctrl_6W_analysed.h5ad")
adata.var_names_make_unique()

In [3]:
adata.obs["type"] = adata.obs["type"].map({"Ctrl":"Ctrl","Reha6W":"Ex","SED6W":"SED"})

## Learn curve using ElPiGraph algorithm

In [4]:
scf.tl.curve(adata,Nodes=20,use_rep="X_umap",epg_mu=0.1,epg_lambda=0.2,ndims_rep=15,seed=100)

inferring a principal curve --> parameters used 
    20 principal points, mu = 0.1, lambda = 0.2
    finished (0:00:02) --> added 
    .uns['epg'] dictionnary containing inferred elastic curve generated from elpigraph.
    .obsm['X_R'] soft assignment of cells to principal points.
    .uns['graph']['B'] adjacency matrix of the principal points.
    .uns['graph']['F'], coordinates of principal points in representation space.


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [5]:
adata.obsm['X_umap']

array([[5.224971 , 2.5934334],
       [5.005595 , 2.0837278],
       [5.365646 , 2.7766287],
       ...,
       [5.3342156, 2.5648618],
       [5.813837 , 2.0333977],
       [5.823919 , 2.3944068]], dtype=float32)

## Plotting the tree

In [6]:
scf.pl.graph(adata)

In [7]:
scf.tl.root(adata,0)

node 0 selected as a root --> added
    .uns['graph']['root'] selected root.
    .uns['graph']['pp_info'] for each PP, its distance vs root and segment assignment.
    .uns['graph']['pp_seg'] segments network information.


In [8]:
scf.tl.pseudotime(adata,n_jobs=20,n_map=100,seed=1)

projecting cells onto the principal graph
    mappings: 100%|███████████████████████████| 100/100 [00:55<00:00,  1.81it/s]
    finished (0:00:57) --> added
    .obs['edge'] assigned edge.
    .obs['t'] pseudotime value.
    .obs['seg'] segment of the tree assigned.
    .obs['milestones'] milestone assigned.
    .uns['pseudotime_list'] list of cell projection from all mappings.


In [9]:
sc.pl.umap(adata,color="t")

In [10]:
scf.pl.trajectory(adata,basis="umap",arrows=False,arrow_offset=3)

In [11]:
sc.pl.violin(adata,"t",groupby="type",rotation=90, size=0)

In [12]:
sc.pl.violin(adata,"t",groupby="leiden",rotation=90, size=0)

In [13]:
scf.tl.test_association(adata,n_jobs=5)

test features for association with the trajectory
    single mapping : 100%|████████████████| 24225/24225 [23:21<00:00, 17.29it/s]
    found 174 significant features (0:23:21) --> added
    .var['p_val'] values from statistical test.
    .var['fdr'] corrected values from multiple testing.
    .var['st'] proportion of mapping in which feature is significant.
    .var['A'] amplitue of change of tested feature.
    .var['signi'] feature is significantly changing along pseudotime.
    .uns['stat_assoc_list'] list of fitted features on the graph for all mappings.


In [14]:
scf.tl.test_association(adata,reapply_filters=True,A_cut=.5)
scf.pl.test_association(adata)

reapplied filters, 1195 significant features


In [15]:
scf.tl.fit(adata,n_jobs=5)

fit features associated with the trajectory
    single mapping : 100%|██████████████████| 1195/1195 [02:12<00:00,  8.99it/s]
    finished (adata subsetted to keep only fitted features!) (0:02:15) --> added
    .layers['fitted'], fitted features on the trajectory for all mappings.
    .raw, unfiltered data.


In [70]:
sc.pl.violin(adata,["Fhl2"],use_raw=True)

In [64]:
scf.pl.single_trend(adata,"Fhl2",basis="umap",color_exp="k",)

In [16]:
adata.var["clusters"]

2610203C22Rik    1
Arfgef1          0
Ncoa2            0
Gm28376          1
Ankrd23          3
                ..
Pfkfb1           0
Rps6ka3          0
Pdha1            2
Arhgap6          0
Mid1             0
Name: clusters, Length: 1195, dtype: category
Categories (5, object): ['0', '1', '2', '3', '4']

In [19]:
adata.var[adata.var.index=="Fhl2"]

,n_cells,mt,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,highly_variable,means,dispersions,dispersions_norm,mean,std,p_val,A,fdr,st,signi,clusters
Fhl2,15343,False,15343,10.257375,71.245174,547313,False,4.238366,3.526822,0.88746,-2.559184e-16,0.902663,4.377417e-105,0.591579,1.060429e-100,1,True,4


In [20]:
adata.var[adata.var.index=="Ppargc1a"]

,n_cells,mt,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,highly_variable,means,dispersions,dispersions_norm,mean,std,p_val,A,fdr,st,signi,clusters
Ppargc1a,12525,False,12525,1.514431,76.526482,80807,True,2.418611,2.647197,1.304654,1.269749e-15,0.976674,2.605306e-134,0.80361,6.311353e-130,1,True,4


In [138]:
adata.var[adata.var.index=="Atp2a2"]

,n_cells,mt,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,highly_variable,means,dispersions,dispersions_norm,mean,std,p_val,A,fdr,st,signi,clusters
Atp2a2,16860,False,16860,4.60497,68.402114,245712,False,3.316118,2.50898,-0.339604,-2.756045e-16,0.926618,0.0,1.183617,0.0,1,True,4


In [157]:
del adata.uns["epg"]
adata.write_h5ad("./adata/scfate.h5ad")

In [15]:
adata = sc.read_h5ad("./adata/scfate.h5ad")

In [16]:
scf.pl.single_trend(adata,"Fhl2",basis="umap",color_exp="k",)

In [17]:
adata.obs = adata.obs.loc[:, ~adata.obs.columns.str.startswith("cluster")]

In [18]:
scf.tl.cluster(adata,n_neighbors=10,metric="cosine", resolution=0.6)

Clustering features using fitted layer


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


    finished (0:01:07) --> added 
    .var['clusters'] identified modules.


In [47]:
scf.pl.trends(adata,features=adata.var_names[adata.var.clusters=="0"],basis="umap")

In [48]:
scf.pl.trends(adata,features=adata.var_names[adata.var.clusters=="1"],basis="umap")

In [49]:
scf.pl.trends(adata,features=adata.var_names[adata.var.clusters=="2"],basis="umap", n_features=10, fontsize=12)

In [22]:
plt.rcParams['font.size'] = 20
scf.pl.trends(adata,features=adata.var_names[adata.var.clusters=="3"],basis="umap", n_features=5, fontsize=12)

In [21]:
plt.rcParams['font.size'] = 20
scf.pl.trends(adata,features=adata.var_names[adata.var.clusters=="4"],basis="umap", n_features=10, highlight_features="fdr", fontsize=12)

In [37]:
adata

AnnData object with n_obs × n_vars = 10865 × 1195
    obs: 'sample', 'type', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type', 't', 'seg', 'edge', 't_sd', 'milestones', 'cluster0_score', 'cluster1_score', 'cluster2_score', 'cluster3_score', 'cluster4_score', 'module 0', 'module 1', 'module 2', 'module 3', 'module 4'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std', 'p_val', 'A', 'fdr', 'st', 'signi', 'clusters'
    uns: 'cell_type_colors', 'dendrogram_leiden', 'graph', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'milestones_colors', 'neighbors', 'pca', 'pseudotime_list', 'rank_genes_groups', 'sample_colors', 'scrublet', 'seg_colors', 'stat_assoc_list', 'type_colors', 'umap'
    obsm: 'X_R', 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    layers: 'fitted'
   

In [36]:
from scanpy.tools import score_genes

# 各クラスタの gene module score を obs に追加
for cl, genes in cluster_genes.items():
    score_genes(adata, gene_list=genes, score_name=f"module {cl}")

In [40]:
module_cols = [c for c in adata.obs.columns if c.startswith("module")]
module_cols

['module 0', 'module 1', 'module 2', 'module 3', 'module 4']

In [48]:
sc.pl.dotplot(
    adata,
    var_names=module_cols,
    groupby="leiden",
    standard_scale="var",   # ← 推奨：列ごとz-score化
    cmap="RdBu_r"
)

In [28]:
# すべてのクラスタごとに辞書形式でまとめる
cluster_genes = {
    str(c): adata.var_names[adata.var["clusters"] == c].tolist()
    for c in adata.var["clusters"].cat.categories
}

In [29]:
with open("cluster_genes.json", "w") as f:
    json.dump(cluster_genes, f, indent=2)

In [30]:
# 遺伝子名を出力
# 最大長を取得
max_len = max(len(genes) for genes in cluster_genes.values())

# 各クラスタのリストを同じ長さに揃える（不足分は空文字で埋める）
for key in cluster_genes:
    cluster_genes[key] += [""] * (max_len - len(cluster_genes[key]))

# DataFrameに変換
df = pd.DataFrame(cluster_genes)

# CSVに保存（必要に応じてパスを変更）
output_path = "cluster_genes.csv"
df.to_csv(output_path, index=False)

print(f"CSV file saved to: {output_path}")

CSV file saved to: cluster_genes.csv


In [57]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# 1. 共通データ作成
# =========================
df = adata.obs[["t","type"] + module_cols].copy()
df = df.sort_values("t").reset_index(drop=True)

# --- Z-score module ---
df[module_cols] = (df[module_cols] - df[module_cols].mean()) / df[module_cols].std()
zmax = 1.5
df[module_cols] = df[module_cols].clip(-zmax, zmax)

# --- 共通 pseudotime bin（rank方式）
n_bin = 100
df["t_rank"] = df["t"].rank(method="first")
df["bin"] = pd.qcut(df["t_rank"], n_bin, labels=False)

# --- binごとの平均module ---
df_mod = df.groupby("bin")[module_cols + ["t"]].mean()

# --- sample内 fraction ---
count = (
    df.groupby(["type","bin"])
    .size()
    .unstack(fill_value=0)
)
frac = count.div(count.sum(axis=1), axis=0)

# --- pseudotime 軸用（bin→t）
bin_to_t = df.groupby("bin")["t"].mean()

# =========================
# bin tick
# =========================
xticks = np.arange(0, n_bin, 20)


# =========================
# 2. figure
# =========================
fig = plt.figure(figsize=(10,5))
gs = fig.add_gridspec(2,1,height_ratios=[8,3],hspace=0.05)

# =========================
# 上：module heatmap
# =========================
ax1 = fig.add_subplot(gs[0])

sns.heatmap(
    df_mod[module_cols].T,
    cmap="RdBu_r",
    linewidths=0,
    vmin=-zmax,
    vmax=zmax,
    ax=ax1,
    cbar_kws={'label': 'Z-score'}
)

ax1.set_ylabel("Module")
ax1.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax1.set_xlabel(None)
# =========================
# 下：sample enrichment heatmap
# =========================
ax2 = fig.add_subplot(gs[1], sharex=ax1)

sns.heatmap(
    frac,
    cmap="rocket",
    linewidths=0,
    ax=ax2,
    cbar_kws={'label': 'Fraction within sample'}
)

ax1.grid(False)
ax2.grid(False)
ax2.set_ylabel("Sample")
ax2.set_xlabel("Pseudotime bin")



# =========================
# pseudotime 目盛りを一致
# =========================

ax2.set_xticks(xticks+0.5)
ax2.set_xticklabels(xticks)

plt.tight_layout()
plt.show()

In [59]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# 1. data preparation
# =========================

df = adata.obs[["t","type"] + module_cols].copy()
df = df.sort_values("t").reset_index(drop=True)

# --- Z-score ---
df[module_cols] = (df[module_cols] - df[module_cols].mean()) / df[module_cols].std()
zmax = 1.5
df[module_cols] = df[module_cols].clip(-zmax, zmax)

# --- pseudotime bin ---
n_bin = 100
df["t_rank"] = df["t"].rank(method="first")
df["bin"] = pd.qcut(df["t_rank"], n_bin, labels=False)

# --- module mean ---
df_mod = df.groupby("bin")[module_cols].mean()

# --- sample fraction ---
count = (
    df.groupby(["type","bin"])
    .size()
    .unstack(fill_value=0)
)

frac = count.div(count.sum(axis=1), axis=0)

# bin順に揃える
frac = frac.reindex(columns=df_mod.index)

# =========================
# 2. figure layout
# =========================

fig = plt.figure(figsize=(10,5))

gs = fig.add_gridspec(
    2,2,
    width_ratios=[40,1],
    height_ratios=[8,3],
    hspace=0.05,
    wspace=0.05
)

ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[1,0], sharex=ax1)

cax1 = fig.add_subplot(gs[0,1])
cax2 = fig.add_subplot(gs[1,1])

# =========================
# 3. module heatmap
# =========================

sns.heatmap(
    df_mod[module_cols].T,
    cmap="RdBu_r",
    vmin=-zmax,
    vmax=zmax,
    linewidths=0,
    ax=ax1,
    cbar_ax=cax1,
    cbar_kws={"label":"Z-score"}
)

ax1.set_ylabel("Module")
ax1.set_xlabel(None)
ax1.tick_params(axis='x', bottom=False, labelbottom=False)
ax1.grid(False)

# =========================
# 4. sample enrichment heatmap
# =========================

sns.heatmap(
    frac,
    cmap="rocket",
    linewidths=0,
    ax=ax2,
    cbar_ax=cax2,
    cbar_kws={"label":"Fraction"}
)

ax2.set_ylabel("Sample")
ax2.set_xlabel("Pseudotime bin")
ax2.grid(False)

# =========================
# 5. x-axis ticks (bin)
# =========================

xticks = np.arange(0, n_bin, 20)

ax2.set_xticks(xticks + 0.5)
ax2.set_xticklabels(xticks)

# =========================
# 6. finish
# =========================

plt.tight_layout()
plt.show()

In [61]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# 1. data preparation
# =========================

df = adata.obs[["t","leiden"] + module_cols].copy()
df = df.sort_values("t").reset_index(drop=True)

# --- Z-score ---
df[module_cols] = (df[module_cols] - df[module_cols].mean()) / df[module_cols].std()
zmax = 1.5
df[module_cols] = df[module_cols].clip(-zmax, zmax)

# --- pseudotime bin ---
n_bin = 100
df["t_rank"] = df["t"].rank(method="first")
df["bin"] = pd.qcut(df["t_rank"], n_bin, labels=False)

# --- module mean ---
df_mod = df.groupby("bin")[module_cols].mean()

# --- sample fraction ---
count = (
    df.groupby(["leiden","bin"])
    .size()
    .unstack(fill_value=0)
)

frac = count.div(count.sum(axis=1), axis=0)

# bin順に揃える
frac = frac.reindex(columns=df_mod.index)

# =========================
# 2. figure layout
# =========================

fig = plt.figure(figsize=(10,5))

gs = fig.add_gridspec(
    2,2,
    width_ratios=[40,1],
    height_ratios=[8,3],
    hspace=0.05,
    wspace=0.05
)

ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[1,0], sharex=ax1)

cax1 = fig.add_subplot(gs[0,1])
cax2 = fig.add_subplot(gs[1,1])

# =========================
# 3. module heatmap
# =========================

sns.heatmap(
    df_mod[module_cols].T,
    cmap="RdBu_r",
    vmin=-zmax,
    vmax=zmax,
    linewidths=0,
    ax=ax1,
    cbar_ax=cax1,
    cbar_kws={"label":"Z-score"}
)

ax1.set_ylabel("Module")
ax1.set_xlabel(None)
ax1.tick_params(axis='x', bottom=False, labelbottom=False)
ax1.grid(False)

# =========================
# 4. sample enrichment heatmap
# =========================

sns.heatmap(
    frac,
    cmap="rocket",
    linewidths=0,
    ax=ax2,
    cbar_ax=cax2,
    cbar_kws={"label":"Fraction"}
)

ax2.set_ylabel("Leiden")
ax2.set_xlabel("Pseudotime bin")
ax2.grid(False)

# =========================
# 5. x-axis ticks (bin)
# =========================

xticks = np.arange(0, n_bin, 20)

ax2.set_xticks(xticks + 0.5)
ax2.set_xticklabels(xticks)

# =========================
# 6. finish
# =========================

plt.tight_layout()
plt.show()

In [82]:
import gseapy as gp

results = {}
geneset_list = ['MSigDB_Hallmark_2020','GO_Biological_Process_2023','KEGG_2019_Mouse','Reactome_2022']


for clust, genes in cluster_genes.items():
    enr = gp.enrichr(
        gene_list = genes,
        gene_sets = geneset_list,
        organism = "Mouse",
        outdir = f"enrichr_cluster{clust}",
        cutoff = 0.05,
        no_plot = True 
    )
    results[clust] = enr.results


In [ ]:
results

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

metrics_to_sort = '-log10(adjusted P-value)'
n_rank = 10   # 可視化する上位件数

for clust, enr_result in results.items():
    print(f"\n==============================")
    print(f" Cluster {clust}")
    print(f"==============================")
    
    for geneset in geneset_list:
        print(f"\n--- {geneset} ---")
        
        # enrichr結果から対象Gene Setを抽出
        df = enr_result[enr_result['Gene_set'] == geneset].copy()
        df = df[df['Adjusted P-value'] < 0.1]
        
        if df.empty:
            print("  → 有意な結果なし")
            continue
        
        # -log10(p)を計算
        df[metrics_to_sort] = -np.log10(df['Adjusted P-value'])
        df = df.sort_values(metrics_to_sort, ascending=False)
        
        # 上位30を表示
        display(df.head(10))
        
        # 可視化
        plt.rcParams['axes.grid'] = False
        plt.rcParams['figure.figsize'] = (6, 6)
        plt.barh(
            width=df.head(n_rank)[metrics_to_sort],
            y=[x.split(' (')[0] for x in df.head(n_rank)['Term']],
            color='darkred'
        )
        plt.yticks(fontsize=20) 
        plt.tick_params(axis="y", labelsize=15)
        plt.gca().invert_yaxis()
        plt.xlabel(metrics_to_sort)
        plt.title(f'Cluster {clust} - {geneset}')
        plt.margins(y=0.02)
        plt.show()



 Cluster 0

--- MSigDB_Hallmark_2020 ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,-log10(adjusted P-value)
0,MSigDB_Hallmark_2020,UV Response Dn,20/144,7.505392e-11,3.377426e-09,0,0,7.338899,171.090379,NRP1;PRKCE;MAGI2;BCKDHB;PTPRM;PRKCA;SYNJ2;NR3C...,8.471414
1,MSigDB_Hallmark_2020,Mitotic Spindle,13/199,5.615245e-04,1.263430e-02,0,0,3.118911,23.344600,ARFGEF1;ROCK1;EPB41;PCGF5;ITSN1;MID1;AKAP13;FG...,1.898449
2,MSigDB_Hallmark_2020,Adipogenesis,12/200,1.873595e-03,2.810393e-02,0,0,2.841526,17.844488,SLC27A1;ACAA2;ACADL;REEP5;ANGPT1;IDH1;ITSN1;AC...,1.551233
3,MSigDB_Hallmark_2020,Fatty Acid Metabolism,10/158,2.982286e-03,3.355071e-02,0,0,3.000340,17.447174,HADHB;CPT1A;ACAA2;ACADL;ACSL1;IDH1;BCKDHB;ACAD...,1.474298



--- GO_Biological_Process_2023 ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,-log10(adjusted P-value)
45,GO_Biological_Process_2023,Regulation Of Cardiac Muscle Cell Contraction ...,9/26,2.863764e-09,0.000006,0,0,23.613215,464.498601,DSP;PLN;JUP;PDE4D;PDE4B;CTNNA3;ANK2;DSG2;FGF13,5.215908
46,GO_Biological_Process_2023,Positive Regulation Of Epithelial Cell Migrati...,14/95,2.487950e-08,0.000026,0,0,7.772589,136.091976,ANGPT1;PRKCE;PRKCA;PROX1;FGF1;HIF1A;EPB41L4B;S...,4.578034
47,GO_Biological_Process_2023,Cardiac Conduction (GO:0061337),10/46,5.582164e-08,0.000040,0,0,12.405924,207.192635,DSP;MEF2A;CACNB2;JUP;KCND3;KCNQ1;CTNNA3;ANK2;D...,4.403164
49,GO_Biological_Process_2023,Bundle Of His Cell To Purkinje Myocyte Communi...,5/8,2.889245e-07,0.000123,0,0,73.717949,1109.978550,DSP;JUP;DSG2;TNNI3K;TBX5,3.911031
48,GO_Biological_Process_2023,Regulation Of Heart Rate By Cardiac Conduction...,9/41,2.384825e-07,0.000123,0,0,12.534889,191.144143,DSP;CACNB2;JUP;KCND3;KCNQ1;CTNNA3;ANK2;DSG2;KCNJ3,3.911031
50,GO_Biological_Process_2023,Positive Regulation Of Transcription By RNA Po...,46/938,5.210607e-07,0.000184,0,0,2.399847,34.719542,HDAC4;THRA;ZMYND8;GATA6;GATA4;RORA;FGF1;NR3C1;...,3.734108
51,GO_Biological_Process_2023,Atrial Cardiac Muscle Cell Action Potential (G...,5/12,3.794106e-06,0.001151,0,0,31.586942,394.270163,CACNB2;GJA1;KCNQ1;KCNN2;KCNJ3,2.938834
52,GO_Biological_Process_2023,Positive Regulation Of DNA-templated Transcrip...,52/1243,9.907646e-06,0.002630,0,0,2.029619,23.385681,THRA;RORA;FGF1;NR3C1;RPS6KA3;LBH;RPS6KA5;ZMIZ1...,2.579965
53,GO_Biological_Process_2023,Cardiac Myofibril Assembly (GO:0055003),5/15,1.360878e-05,0.003212,0,0,22.107466,247.709633,FHOD3;MEF2A;NEBL;PROX1;MYLK3,2.493269
54,GO_Biological_Process_2023,Negative Regulation Of Sodium Ion Transmembran...,4/9,2.837707e-05,0.005774,0,0,35.301129,369.600316,NEDD4;PRKCE;STK39;NEDD4L,2.238515



--- KEGG_2019_Mouse ---


...

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,-log10(adjusted P-value)
20,GO_Biological_Process_2023,Regulation Of Heart Contraction (GO:0008016),7/69,7.572912e-10,3.990925e-07,0,0,46.767809,982.183538,DMPK;KCNIP2;TNNT2;RNF207;ATP2A2;SLC4A3;MYH6,6.398926
21,GO_Biological_Process_2023,Muscle Contraction (GO:0006936),6/94,2.208252e-07,5.818744e-05,0,0,27.630334,423.459579,SMTN;PABPN1;KCNIP2;TNNT2;MYH6;TTN,4.235171
22,GO_Biological_Process_2023,Striated Muscle Contraction (GO:0006941),5/57,4.902911e-07,8.612781e-05,0,0,38.255769,555.790010,MYBPC3;MYH7B;TNNT2;MYH6;TTN,4.064857
23,GO_Biological_Process_2023,RNA Processing (GO:0006396),7/183,6.704740e-07,8.833495e-05,0,0,16.380563,232.854311,DDX17;SON;PABPN1;AKAP8L;LUC7L3;PPARGC1A;SRRM1,4.053867
24,GO_Biological_Process_2023,Cardiac Muscle Tissue Morphogenesis (GO:0055008),4/31,1.523701e-06,1.470382e-04,0,0,57.859114,774.986299,MYBPC3;TNNT2;MYH6;TTN,3.832570
25,GO_Biological_Process_2023,Cardiac Muscle Contraction (GO:0060048),4/33,1.973499e-06,1.470382e-04,0,0,53.863421,707.533863,MYBPC3;TNNT2;MYH6;TTN,3.832570
26,GO_Biological_Process_2023,Actomyosin Structure Organization (GO:0031032),5/77,2.219159e-06,1.470382e-04,0,0,27.601389,359.325431,MYH7B;TNNT2;MYO18A;MYH6;TTN,3.832570
27,GO_Biological_Process_2023,Regulation Of Blood Circulation (GO:1903522),4/34,2.232078e-06,1.470382e-04,0,0,52.065359,677.504523,DMPK;KCNIP2;TNNT2;MYH6,3.832570
28,GO_Biological_Process_2023,Heart Contraction (GO:0060047),4/41,4.805081e-06,2.813642e-04,0,0,42.200318,516.778203,MYBPC3;TNNT2;MYH6;TTN,3.550731
29,GO_Biological_Process_2023,"Regulation Of mRNA Splicing, Via Spliceosome (...",5/94,5.950824e-06,3.136084e-04,0,0,22.310112,268.434844,DDX17;RBM25;SON;RBM20;SRRM1,3.503612



--- KEGG_2019_Mouse ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,-log10(adjusted P-value)
547,KEGG_2019_Mouse,Hypertrophic cardiomyopathy (HCM),6/86,1.296736e-07,0.000005,0,0,30.405612,482.179650,MYBPC3;TNNT2;ATP2A2;MYH6;ACTG1;TTN,5.277459
548,KEGG_2019_Mouse,Dilated cardiomyopathy (DCM),6/90,1.702860e-07,0.000005,0,0,28.951895,451.238054,MYBPC3;TNNT2;ATP2A2;MYH6;ACTG1;TTN,5.277459
549,KEGG_2019_Mouse,Glucagon signaling pathway,5/102,8.880636e-06,0.000184,0,0,20.461856,238.004884,PYGM;PPARA;ACACB;PPARGC1A;CPT1B,3.736285
550,KEGG_2019_Mouse,Insulin resistance,5/110,1.283398e-05,0.000199,0,0,18.895238,212.824887,PYGM;PPARA;ACACB;PPARGC1A;CPT1B,3.701307
551,KEGG_2019_Mouse,Adipocytokine signaling pathway,4/71,4.336939e-05,0.000538,0,0,23.269535,233.760084,PPARA;ACACB;PPARGC1A;CPT1B,3.269395
552,KEGG_2019_Mouse,Thyroid hormone signaling pathway,4/115,2.821654e-04,0.002916,0,0,14.014485,114.540627,NCOR1;ATP2A2;MYH6;ACTG1,2.535256
553,KEGG_2019_Mouse,Cardiac muscle contraction,3/78,1.293825e-03,0.011460,0,0,15.284615,101.645025,TNNT2;ATP2A2;MYH6,1.940831
554,KEGG_2019_Mouse,mRNA surveillance pathway,3/96,2.346709e-03,0.018187,0,0,12.315136,74.564968,PABPN1;MSI2;SRRM1,1.740239
555,KEGG_2019_Mouse,Thermogenesis,4/231,3.729594e-03,0.025693,0,0,6.812819,38.093575,PRDM16;PPARGC1A;CPT1B;ACTG1,1.590189
556,KEGG_2019_Mouse,Pyruvate metabolism,2/38,4.899012e-03,0.028436,0,0,20.868973,110.996257,ACYP2;ACACB,1.546137



--- Reactome_2022 ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,-log10(adjusted P-value)
609,Reactome_2022,Transcriptional Activation Of Mitochondrial Bi...,4/52,0.000013,0.001444,0,0,32.511438,366.889563,NCOR1;IDH2;PPARA;PPARGC1A,2.840496
610,Reactome_2022,Muscle Contraction R-HSA-397014,6/196,0.000016,0.001444,0,0,12.731472,140.629709,MYBPC3;DMPK;KCNIP2;TNNT2;ATP2A2;MYH6,2.840496
611,Reactome_2022,RHOBTB2 GTPase Cycle R-HSA-9013418,3/23,0.000034,0.002022,0,0,57.475962,592.201589,MSI2;ACTG1;SRRM1,2.694181
612,Reactome_2022,Striated Muscle Contraction R-HSA-390522,3/33,0.000101,0.003650,0,0,38.298077,352.256603,MYBPC3;TNNT2;MYH6,2.437665
613,Reactome_2022,Mitochondrial Biogenesis R-HSA-1592230,4/89,0.000105,0.003650,0,0,18.325260,167.876748,NCOR1;IDH2;PPARA;PPARGC1A,2.437665
614,Reactome_2022,RHOBTB GTPase Cycle R-HSA-9706574,3/35,0.000121,0.003650,0,0,35.900841,323.813694,MSI2;ACTG1;SRRM1,2.437665
615,Reactome_2022,Heme Signaling R-HSA-9707616,3/45,0.000257,0.006261,0,0,27.339286,225.966935,NCOR1;PPARA;PPARGC1A,2.203383
616,Reactome_2022,PPARA Activates Gene Expression R-HSA-1989781,4/116,0.000292,0.006261,0,0,13.888655,113.053065,NCOR1;FHL2;PPARA;PPARGC1A,2.203383
617,Reactome_2022,Regulation Of Lipid Metabolism By PPARalpha R-...,4/118,0.000311,0.006261,0,0,13.643619,110.168789,NCOR1;FHL2;PPARA;PPARGC1A,2.203383
618,Reactome_2022,mRNA 3-End Processing R-HSA-72187,3/58,0.000546,0.009874,0,0,20.863636,156.764148,PABPN1;SRSF11;SRRM1,2.005502


In [86]:
import pandas as pd

metrics_to_sort = 'Combined Score'

# 保存用のリスト
all_results = []

for clust, enr_result in results.items():
    for geneset in geneset_list:
        # enrichr結果から対象Gene Setを抽出
        df = enr_result[enr_result['Gene_set'] == geneset].copy()
        df = df[df['Adjusted P-value'] < 0.1]  # 閾値フィルタ

        if df.empty:
            continue

        # 必要な列を残す
        df = df[['Term', 'Adjusted P-value', 'Combined Score', "Genes"]].copy()
        df['Cluster'] = clust
        df['GeneSet'] = geneset

        # リストに追加
        all_results.append(df)

# まとめて DataFrame に
if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)

    # 並び替え（例: Cluster → GeneSet → Combined Score 降順）
    combined_df = combined_df.sort_values(
        by=['Cluster', 'GeneSet', metrics_to_sort],
        ascending=[True, True, False]
    )

    # CSVに保存
    output_path = "enrichment_results_summary.csv"
    combined_df.to_csv(output_path, index=False)
    print(f"CSV file saved to: {output_path}")

    # 上位部分を表示
    display(combined_df.head(20))
else:
    print("有意な結果は見つかりませんでした。")


CSV file saved to: enrichment_results_summary.csv


,Term,Adjusted P-value,Combined Score,Genes,Cluster,GeneSet
8,Bundle Of His Cell To Purkinje Myocyte Communi...,0.000123,1109.978550,DSP;JUP;DSG2;TNNI3K;TBX5,0,GO_Biological_Process_2023
19,Bundle Of His cell-Purkinje Myocyte Adhesion I...,0.013319,603.728486,DSP;JUP;DSG2,0,GO_Biological_Process_2023
20,Regulation Of Relaxation Of Cardiac Muscle (GO...,0.013319,603.728486,PLN;PDE4D;PDE4B,0,GO_Biological_Process_2023
4,Regulation Of Cardiac Muscle Cell Contraction ...,0.000006,464.498601,DSP;PLN;JUP;PDE4D;PDE4B;CTNNA3;ANK2;DSG2;FGF13,0,GO_Biological_Process_2023
10,Atrial Cardiac Muscle Cell Action Potential (G...,0.001151,394.270163,CACNB2;GJA1;KCNQ1;KCNN2;KCNJ3,0,GO_Biological_Process_2023
27,Cardiac Muscle Cell-Cardiac Muscle Cell Adhesi...,0.017921,372.682278,DSP;JUP;DSG2,0,GO_Biological_Process_2023
28,Desmosome Organization (GO:0002934),0.017921,372.682278,DSP;JUP;PRKCA,0,GO_Biological_Process_2023
13,Negative Regulation Of Sodium Ion Transmembran...,0.005774,369.600316,NEDD4;PRKCE;STK39;NEDD4L,0,GO_Biological_Process_2023
15,Negative Regulation Of Sodium Ion Transmembran...,0.008223,293.482003,NEDD4;PRKCE;STK39;NEDD4L,0,GO_Biological_Process_2023
12,Cardiac Myofibril Assembly (GO:0055003),0.003212,247.709633,FHOD3;MEF2A;NEBL;PROX1;MYLK3,0,GO_Biological_Process_2023


In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

metrics_to_sort = 'Combined Score'
n_rank = 10   # 可視化する上位件数

for clust, enr_result in results.items():
    print(f"\n==============================")
    print(f" Cluster {clust}")
    print(f"==============================")
    
    for geneset in geneset_list:
        print(f"\n--- {geneset} ---")
        
        # enrichr結果から対象Gene Setを抽出
        df = enr_result[enr_result['Gene_set'] == geneset].copy()
        df = df[df['Adjusted P-value'] < 0.1]
        
        if df.empty:
            print("  → 有意な結果なし")
            continue
        
        # -log10(p)を計算
        df[metrics_to_sort] = df['Combined Score']
        df = df.sort_values(metrics_to_sort, ascending=False)
        
        # 上位30を表示
        display(df.head(10))
        
        # 可視化
        plt.rcParams['axes.grid'] = False
        plt.rcParams['figure.figsize'] = (6, 6)
        plt.barh(
            width=df.head(n_rank)[metrics_to_sort],
            y=[x.split(' (')[0] for x in df.head(n_rank)['Term']],
            color='darkred'
        )
        plt.yticks(fontsize=20) 
        plt.tick_params(axis="y", labelsize=18)
        plt.gca().invert_yaxis()
        plt.xlabel(metrics_to_sort)
        plt.title(f'Cluster {clust} - {geneset}')
        plt.margins(y=0.02)
        plt.show()



 Cluster 0

--- MSigDB_Hallmark_2020 ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
0,MSigDB_Hallmark_2020,UV Response Dn,20/144,7.505392e-11,3.377426e-09,0,0,7.338899,171.090379,NRP1;PRKCE;MAGI2;BCKDHB;PTPRM;PRKCA;SYNJ2;NR3C...
1,MSigDB_Hallmark_2020,Mitotic Spindle,13/199,5.615245e-04,1.263430e-02,0,0,3.118911,23.344600,ARFGEF1;ROCK1;EPB41;PCGF5;ITSN1;MID1;AKAP13;FG...
2,MSigDB_Hallmark_2020,Adipogenesis,12/200,1.873595e-03,2.810393e-02,0,0,2.841526,17.844488,SLC27A1;ACAA2;ACADL;REEP5;ANGPT1;IDH1;ITSN1;AC...
3,MSigDB_Hallmark_2020,Fatty Acid Metabolism,10/158,2.982286e-03,3.355071e-02,0,0,3.000340,17.447174,HADHB;CPT1A;ACAA2;ACADL;ACSL1;IDH1;BCKDHB;ACAD...



--- GO_Biological_Process_2023 ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
49,GO_Biological_Process_2023,Bundle Of His Cell To Purkinje Myocyte Communi...,5/8,2.889245e-07,0.000123,0,0,73.717949,1109.978550,DSP;JUP;DSG2;TNNI3K;TBX5
61,GO_Biological_Process_2023,Regulation Of Relaxation Of Cardiac Muscle (GO...,3/5,1.072462e-04,0.013319,0,0,66.050676,603.728486,PLN;PDE4D;PDE4B
60,GO_Biological_Process_2023,Bundle Of His cell-Purkinje Myocyte Adhesion I...,3/5,1.072462e-04,0.013319,0,0,66.050676,603.728486,DSP;JUP;DSG2
45,GO_Biological_Process_2023,Regulation Of Cardiac Muscle Cell Contraction ...,9/26,2.863764e-09,0.000006,0,0,23.613215,464.498601,DSP;PLN;JUP;PDE4D;PDE4B;CTNNA3;ANK2;DSG2;FGF13
51,GO_Biological_Process_2023,Atrial Cardiac Muscle Cell Action Potential (G...,5/12,3.794106e-06,0.001151,0,0,31.586942,394.270163,CACNB2;GJA1;KCNQ1;KCNN2;KCNJ3
68,GO_Biological_Process_2023,Cardiac Muscle Cell-Cardiac Muscle Cell Adhesi...,3/6,2.109296e-04,0.017921,0,0,44.031532,372.682278,DSP;JUP;DSG2
69,GO_Biological_Process_2023,Desmosome Organization (GO:0002934),3/6,2.109296e-04,0.017921,0,0,44.031532,372.682278,DSP;JUP;PRKCA
54,GO_Biological_Process_2023,Negative Regulation Of Sodium Ion Transmembran...,4/9,2.837707e-05,0.005774,0,0,35.301129,369.600316,NEDD4;PRKCE;STK39;NEDD4L
56,GO_Biological_Process_2023,Negative Regulation Of Sodium Ion Transmembran...,4/10,4.646010e-05,0.008223,0,0,29.416102,293.482003,NEDD4;PRKCE;STK39;NEDD4L
53,GO_Biological_Process_2023,Cardiac Myofibril Assembly (GO:0055003),5/15,1.360878e-05,0.003212,0,0,22.107466,247.709633,FHOD3;MEF2A;NEBL;PROX1;MYLK3



--- KEGG_2019_Mouse ---


...

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
35,GO_Biological_Process_2023,Brown Fat Cell Differentiation (GO:0050873),2/8,2.057099e-04,5.807418e-03,0,0,125.402516,1064.547426,PRDM16;PPARGC1A
37,GO_Biological_Process_2023,Positive Regulation Of ATP Biosynthetic Proces...,2/8,2.057099e-04,5.807418e-03,0,0,125.402516,1064.547426,PPARA;PPARGC1A
36,GO_Biological_Process_2023,Regulation Of Skeletal Muscle Cell Differentia...,2/8,2.057099e-04,5.807418e-03,0,0,125.402516,1064.547426,DDX17;MYOCD
20,GO_Biological_Process_2023,Regulation Of Heart Contraction (GO:0008016),7/69,7.572912e-10,3.990925e-07,0,0,46.767809,982.183538,DMPK;KCNIP2;TNNT2;RNF207;ATP2A2;SLC4A3;MYH6
39,GO_Biological_Process_2023,Positive Regulation Of ATP Metabolic Process (...,2/9,2.640179e-04,6.893972e-03,0,0,107.482480,885.601213,PPARA;PPARGC1A
24,GO_Biological_Process_2023,Cardiac Muscle Tissue Morphogenesis (GO:0055008),4/31,1.523701e-06,1.470382e-04,0,0,57.859114,774.986299,MYBPC3;TNNT2;MYH6;TTN
43,GO_Biological_Process_2023,Muscle Filament Sliding (GO:0030049),2/10,3.294406e-04,7.233966e-03,0,0,94.042453,754.043162,TNNT2;MYH6
41,GO_Biological_Process_2023,Negative Regulation Of Glycolytic Process (GO:...,2/10,3.294406e-04,7.233966e-03,0,0,94.042453,754.043162,NCOR1;PPARA
42,GO_Biological_Process_2023,Positive Regulation Of Voltage-Gated Potassium...,2/10,3.294406e-04,7.233966e-03,0,0,94.042453,754.043162,KCNIP2;RNF207
25,GO_Biological_Process_2023,Cardiac Muscle Contraction (GO:0060048),4/33,1.973499e-06,1.470382e-04,0,0,53.863421,707.533863,MYBPC3;TNNT2;MYH6;TTN



--- KEGG_2019_Mouse ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
547,KEGG_2019_Mouse,Hypertrophic cardiomyopathy (HCM),6/86,1.296736e-07,0.000005,0,0,30.405612,482.179650,MYBPC3;TNNT2;ATP2A2;MYH6;ACTG1;TTN
548,KEGG_2019_Mouse,Dilated cardiomyopathy (DCM),6/90,1.702860e-07,0.000005,0,0,28.951895,451.238054,MYBPC3;TNNT2;ATP2A2;MYH6;ACTG1;TTN
549,KEGG_2019_Mouse,Glucagon signaling pathway,5/102,8.880636e-06,0.000184,0,0,20.461856,238.004884,PYGM;PPARA;ACACB;PPARGC1A;CPT1B
551,KEGG_2019_Mouse,Adipocytokine signaling pathway,4/71,4.336939e-05,0.000538,0,0,23.269535,233.760084,PPARA;ACACB;PPARGC1A;CPT1B
550,KEGG_2019_Mouse,Insulin resistance,5/110,1.283398e-05,0.000199,0,0,18.895238,212.824887,PYGM;PPARA;ACACB;PPARGC1A;CPT1B
552,KEGG_2019_Mouse,Thyroid hormone signaling pathway,4/115,2.821654e-04,0.002916,0,0,14.014485,114.540627,NCOR1;ATP2A2;MYH6;ACTG1
556,KEGG_2019_Mouse,Pyruvate metabolism,2/38,4.899012e-03,0.028436,0,0,20.868973,110.996257,ACYP2;ACACB
553,KEGG_2019_Mouse,Cardiac muscle contraction,3/78,1.293825e-03,0.011460,0,0,15.284615,101.645025,TNNT2;ATP2A2;MYH6
554,KEGG_2019_Mouse,mRNA surveillance pathway,3/96,2.346709e-03,0.018187,0,0,12.315136,74.564968,PABPN1;MSI2;SRRM1
557,KEGG_2019_Mouse,AMPK signaling pathway,3/126,5.045038e-03,0.028436,0,0,9.297373,49.177062,ACACB;PPARGC1A;CPT1B



--- Reactome_2022 ---


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
611,Reactome_2022,RHOBTB2 GTPase Cycle R-HSA-9013418,3/23,0.000034,0.002022,0,0,57.475962,592.201589,MSI2;ACTG1;SRRM1
609,Reactome_2022,Transcriptional Activation Of Mitochondrial Bi...,4/52,0.000013,0.001444,0,0,32.511438,366.889563,NCOR1;IDH2;PPARA;PPARGC1A
612,Reactome_2022,Striated Muscle Contraction R-HSA-390522,3/33,0.000101,0.003650,0,0,38.298077,352.256603,MYBPC3;TNNT2;MYH6
614,Reactome_2022,RHOBTB GTPase Cycle R-HSA-9706574,3/35,0.000121,0.003650,0,0,35.900841,323.813694,MSI2;ACTG1;SRRM1
615,Reactome_2022,Heme Signaling R-HSA-9707616,3/45,0.000257,0.006261,0,0,27.339286,225.966935,NCOR1;PPARA;PPARGC1A
613,Reactome_2022,Mitochondrial Biogenesis R-HSA-1592230,4/89,0.000105,0.003650,0,0,18.325260,167.876748,NCOR1;IDH2;PPARA;PPARGC1A
618,Reactome_2022,mRNA 3-End Processing R-HSA-72187,3/58,0.000546,0.009874,0,0,20.863636,156.764148,PABPN1;SRSF11;SRRM1
610,Reactome_2022,Muscle Contraction R-HSA-397014,6/196,0.000016,0.001444,0,0,12.731472,140.629709,MYBPC3;DMPK;KCNIP2;TNNT2;ATP2A2;MYH6
619,Reactome_2022,RNA Polymerase II Transcription Termination R-...,3/67,0.000832,0.013677,0,0,17.921575,127.087838,PABPN1;SRSF11;SRRM1
620,Reactome_2022,Circadian Clock R-HSA-400253,3/69,0.000907,0.013677,0,0,17.376748,121.735250,NCOR1;PPARA;PPARGC1A


# contribution to cluster classification

In [53]:
adata

AnnData object with n_obs × n_vars = 10865 × 1195
    obs: 'sample', 'type', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type', 't', 'seg', 'edge', 't_sd', 'milestones', 'cluster0_score', 'cluster1_score', 'cluster2_score', 'cluster3_score', 'cluster4_score', 'cluster5_score', 'cluster6_score', 'cluster7_score'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std', 'p_val', 'A', 'fdr', 'st', 'signi', 'clusters'
    uns: 'cell_type_colors', 'dendrogram_leiden', 'graph', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'milestones_colors', 'neighbors', 'pca', 'pseudotime_list', 'rank_genes_groups', 'sample_colors', 'scrublet', 'seg_colors', 'stat_assoc_list', 'type_colors', 'umap'
    obsm: 'X_R', 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    layers: 'fitted'
    obsp:

# Intra sample comparison

In [21]:
adata_deg = sc.read_h5ad("./adata/CM_Ctrl_6W_analysed_raw.h5ad")

In [22]:
sc.pp.normalize_total(adata_deg, target_sum=1e4)

In [23]:
sc.pp.log1p(adata_deg)

In [24]:
adata_deg.var["gene_cluster"] = adata.var["clusters"]

In [25]:
adata_deg.var["gene_cluster_4"] = adata_deg.var_names.isin(cluster_genes["4"])
adata_deg.var["gene_cluster_3"] = adata_deg.var_names.isin(cluster_genes["3"])

In [26]:
adata_deg.var["gene_cluster_4"]

Xkr4              False
Gm1992            False
Gm19938           False
Gm37381           False
Rp1               False
                  ...  
4933409K07Rik     False
Gm10931           False
CT868723.1        False
CAAA01147332.1    False
AC149090.1        False
Name: gene_cluster_4, Length: 24225, dtype: bool

In [27]:
# Obtain cluster-specific differentially expressed gene
sc.tl.rank_genes_groups(adata_deg, mask_var="gene_cluster_4",groupby="leiden",groups=("2","0"),reference="0", method="wilcoxon")

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [28]:
# Extract rank genes groups data from adata
rank_genes_groups = adata_deg.uns["rank_genes_groups"]

# Convert the data into a DataFrame
groups = rank_genes_groups['names'].dtype.names  # Get all group names
dfs = []

for group in groups:
    df = pd.DataFrame({
        'gene': rank_genes_groups['names'][group],
        'logfc': rank_genes_groups['logfoldchanges'][group],
        'pvals_adj': rank_genes_groups['pvals_adj'][group]
    })
    df['group'] = group  # Add group information to identify the cell type/cluster
    dfs.append(df)

# Concatenate all groups into a single DataFrame
DEG = pd.concat(dfs, ignore_index=True)

In [29]:
DEG_leiden = DEG

In [30]:
DEG = DEG.loc[DEG["pvals_adj"] < 0.05]
DEG_2 = DEG.loc[DEG["logfc"] > 0]
DEG_0 = DEG.loc[DEG["logfc"] < 0]

In [31]:
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    figsize=(10, 8),  # Adjust as needed
    fontsize=22
)

In [32]:
import matplotlib.pyplot as plt
import numpy as np
from adjustText import adjust_text

# Sort by adjusted p-value (小さいほど上位)
DEG_sorted = DEG_2.sort_values(by='pvals_adj', ascending=True).copy()
DEG_sorted['neg_log_pvals_adj'] = -np.log10(DEG_sorted['pvals_adj'])

# Create the dot plot
plt.figure(figsize=(10, 8))
plt.scatter(range(len(DEG_sorted)), DEG_sorted['neg_log_pvals_adj'], color='blue', s=50)
plt.ylabel('-log10(Adjusted p-value)')
plt.xlabel('Genes (Index)')
plt.title('Rank Plot of -log10(Adjusted p-value)')

# 上位4遺伝子を取得
top_genes = DEG_sorted.head(4)

# 注釈追加
texts = []
for rank, (i, row) in enumerate(top_genes.iterrows(), start=1):
    # 3位の遺伝子だけ赤文字、他は黒
    color = "red" if rank == 3 else "black"
    # italic 表示 (mathtext で遺伝子名をイタリック)
    text = plt.text(
        i, row['neg_log_pvals_adj'],
        f"${row['gene']}$",   # ← mathtextを使うと italic になる
        fontsize=32,
        color=color
    )
    texts.append(text)

# ラベル位置調整＋矢印
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray'))

# x軸目盛り消す
plt.xticks([])

plt.tight_layout()
plt.show()


In [33]:
import matplotlib.pyplot as plt
import numpy as np

# padj 昇順に並べ替え
DEG_sorted = DEG_2.sort_values(by='pvals_adj', ascending=True).copy()
DEG_sorted['neg_log_pvals_adj'] = -np.log10(DEG_sorted['pvals_adj'])

# 上位10遺伝子を取得
top_genes = DEG_sorted.head(10).reset_index(drop=True)

# 横棒グラフ
plt.figure(figsize=(8, 6))
plt.barh(
    y=range(len(top_genes)),
    width=top_genes['neg_log_pvals_adj']
)

# y 軸ラベルを italic に
plt.yticks(
    range(len(top_genes)),
    [f"${g}$" for g in top_genes['gene']],
    fontsize=14
)

plt.xlabel('-log10(Adjusted p-value)')
plt.title('Top 10 genes by adjusted p-value')
plt.gca().invert_yaxis()  # 上位を上に表示
plt.tight_layout()
plt.show()


In [34]:
# Obtain cluster-specific differentially expressed gene
sc.tl.rank_genes_groups(adata_deg, mask_var="gene_cluster_3",groupby="leiden",groups=("0","2"),reference="2", method="wilcoxon")

In [35]:
# Extract rank genes groups data from adata
rank_genes_groups = adata_deg.uns["rank_genes_groups"]

# Convert the data into a DataFrame
groups = rank_genes_groups['names'].dtype.names  # Get all group names
dfs = []

for group in groups:
    df = pd.DataFrame({
        'gene': rank_genes_groups['names'][group],
        'logfc': rank_genes_groups['logfoldchanges'][group],
        'pvals_adj': rank_genes_groups['pvals_adj'][group]
    })
    df['group'] = group  # Add group information to identify the cell type/cluster
    dfs.append(df)

# Concatenate all groups into a single DataFrame
DEG = pd.concat(dfs, ignore_index=True)

In [36]:
DEG_leiden = DEG

In [37]:
DEG = DEG.loc[DEG["pvals_adj"] < 0.05]
DEG_0 = DEG.loc[DEG["logfc"] > 0]

In [38]:
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    figsize=(10, 8),  # Adjust as needed
    fontsize=22
)

In [39]:
import matplotlib.pyplot as plt
import numpy as np
from adjustText import adjust_text

# Sort by adjusted p-value (小さいほど上位)
DEG_sorted = DEG_0.sort_values(by='pvals_adj', ascending=True).copy()
DEG_sorted['neg_log_pvals_adj'] = -np.log10(DEG_sorted['pvals_adj'])

# Create the dot plot
plt.figure(figsize=(10, 8))
plt.scatter(range(len(DEG_sorted)), DEG_sorted['neg_log_pvals_adj'], color='blue', s=50)
plt.ylabel('-log10(Adjusted p-value)')
plt.xlabel('Genes (Index)')
plt.title('Rank Plot of -log10(Adjusted p-value)')

# 上位4遺伝子を取得
top_genes = DEG_sorted.head(4)

# 注釈追加
texts = []
for rank, (i, row) in enumerate(top_genes.iterrows(), start=1):
    # 3位の遺伝子だけ赤文字、他は黒
    color = "red" if rank == 3 else "black"
    # italic 表示 (mathtext で遺伝子名をイタリック)
    text = plt.text(
        i, row['neg_log_pvals_adj'],
        f"${row['gene']}$",   # ← mathtextを使うと italic になる
        fontsize=32,
        color=color
    )
    texts.append(text)

# ラベル位置調整＋矢印
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray'))

# x軸目盛り消す
plt.xticks([])

plt.tight_layout()
plt.show()


In [40]:
import matplotlib.pyplot as plt
import numpy as np

# padj 昇順に並べ替え
DEG_sorted = DEG_0.sort_values(by='pvals_adj', ascending=True).copy()
DEG_sorted['neg_log_pvals_adj'] = -np.log10(DEG_sorted['pvals_adj'])

# 上位10遺伝子を取得
top_genes = DEG_sorted.head(10).reset_index(drop=True)

# 横棒グラフ
plt.figure(figsize=(8, 6))
plt.barh(
    y=range(len(top_genes)),
    width=top_genes['neg_log_pvals_adj']
)

# y 軸ラベルを italic に
plt.yticks(
    range(len(top_genes)),
    [f"${g}$" for g in top_genes['gene']],
    fontsize=14
)

plt.xlabel('-log10(Adjusted p-value)')
plt.title('Top 10 genes by adjusted p-value')
plt.gca().invert_yaxis()  # 上位を上に表示
plt.tight_layout()
plt.show()


In [41]:
# padj 昇順に並べ替え
DEG_sorted = DEG_0.sort_values(by='pvals_adj', ascending=True).copy()
DEG_sorted['neg_log_pvals_adj'] = -np.log10(DEG_sorted['pvals_adj'])

# 上位10遺伝子を取得
top_genes_cluster3 = DEG_sorted.head(10).reset_index(drop=True)

In [42]:
# padj 昇順に並べ替え
DEG_sorted = DEG_2.sort_values(by='pvals_adj', ascending=True).copy()
DEG_sorted['neg_log_pvals_adj'] = -np.log10(DEG_sorted['pvals_adj'])

# 上位10遺伝子を取得
top_genes_cluster4 = DEG_sorted.head(10).reset_index(drop=True)

In [43]:
gene_list3 = top_genes_cluster3["gene"].tolist()

In [44]:
scf.pl.trends(adata,features=adata.var_names[adata.var.clusters=="3"],basis="umap", highlight_features=gene_list3, fontsize=12)

In [45]:
gene_list4 = top_genes_cluster4["gene"].tolist()

In [46]:
scf.pl.trends(adata,features=adata.var_names[adata.var.clusters=="4"],basis="umap", n_features=10,highlight_features=gene_list4,fontsize=12)